# 🏥 Hospital Management System — SQL Mini Project
**Graded Mini Project | Data Science & Business Analytics Program, IIT Guwahati**

Author: Kanak Baghel · [GitHub](https://github.com/Kanakbaghel/hospital_database_project) · [LinkedIn](https://www.linkedin.com/in/kanakbaghel)

This notebook is the completed, end-to-end version of the project described in the repo README. It:

1. Builds the `Doctors`, `Patients`, and `Appointments` schema (with proper primary/foreign keys).
2. Loads consistent sample data (the original `project_hospital.sql` had orphaned foreign keys — e.g. patients pointing at `doctor_id`s that didn't exist yet — which is fixed here).
3. Answers all the graded questions using JOINs, aggregates, subqueries, and window functions.
4. Visualizes the results.
5. Exports a portable `hospital.db` (SQLite) file that the deployment app (`app.py`) reads from.

**Why SQLite instead of MySQL here?** The logic, schema, and SQL syntax below are MySQL-compatible (this was originally a MySQL project — see the note in Section 1). SQLite is used in the notebook purely so it runs end-to-end with zero setup — no server, no credentials. Section 1 also includes the equivalent MySQL connection code, commented out, so you can point this at a real MySQL server for the submission if your grading rubric expects MySQL specifically.


## 1. Setup & Database Connection

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (8, 4.5)

DB_PATH = "hospital.db"

# Fresh build every run, so the notebook is always reproducible
import os
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")
cur = conn.cursor()
print("Connected to", DB_PATH)


In [ ]:
# --- MySQL equivalent (use this instead of the sqlite3 connection above if you have a MySQL server) ---
# import mysql.connector
# conn = mysql.connector.connect(
#     host="localhost",
#     user="root",
#     password="your_password",
#     database="hospital_db"
# )
# cur = conn.cursor()
# (mysql.connector needs `pip install mysql-connector-python`)


## 2. Schema Design

Same three tables documented in the project README, with foreign keys enforced:

| Table | Columns |
|---|---|
| `Doctors` | doctor_id (PK), doctor_name, specialization, experience_years |
| `Patients` | patient_id (PK), patient_name, age, gender, doctor_id (FK), contact_number |
| `Appointments` | appointment_id (PK), patient_id (FK), doctor_id (FK), appointment_date, status |


In [ ]:
cur.executescript('''
DROP TABLE IF EXISTS Appointments;
DROP TABLE IF EXISTS Patients;
DROP TABLE IF EXISTS Doctors;

CREATE TABLE Doctors (
    doctor_id         INTEGER PRIMARY KEY AUTOINCREMENT,
    doctor_name       VARCHAR(100) NOT NULL,
    specialization    VARCHAR(100) NOT NULL,
    experience_years  INTEGER NOT NULL
);

CREATE TABLE Patients (
    patient_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_name    VARCHAR(100) NOT NULL,
    age             INTEGER NOT NULL,
    gender          VARCHAR(10) NOT NULL,
    doctor_id       INTEGER,
    contact_number  VARCHAR(20),
    FOREIGN KEY (doctor_id) REFERENCES Doctors(doctor_id)
);

CREATE TABLE Appointments (
    appointment_id     INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id         INTEGER,
    doctor_id          INTEGER,
    appointment_date   DATETIME NOT NULL,
    status             VARCHAR(20) NOT NULL,
    FOREIGN KEY (patient_id) REFERENCES Patients(patient_id),
    FOREIGN KEY (doctor_id) REFERENCES Doctors(doctor_id)
);
''')
conn.commit()
print("Tables created: Doctors, Patients, Appointments")


## 3. Sample Data (INSERT)

10 doctors, 12 patients, 15 appointments — enough rows to make every JOIN, GROUP BY, and window
function below produce a non-trivial result (e.g. some doctors with zero patients, some patients
sharing an appointment date, a clear experience ranking).

In [ ]:
doctors = [
    ('Dr. Jessica Fernandes', 'Pathology', 5),
    ('Dr. Karen Mehta', 'Gastroenterology', 8),
    ('Dr. Brian Dsouza', 'Endocrinology', 4),
    ('Dr. George Thomas', 'Nephrology', 6),
    ('Dr. Amanda Rao', 'Psychiatry', 9),
    ('Dr. Nikhil Sharma', 'Cardiology', 12),
    ('Dr. Priya Nair', 'Orthopedics', 15),
    ('Dr. Sameer Khan', 'Dermatology', 3),
    ('Dr. Anita Verma', 'Pediatrics', 11),
    ('Dr. Rahul Gupta', 'Neurology', 2),
]
cur.executemany(
    "INSERT INTO Doctors (doctor_name, specialization, experience_years) VALUES (?,?,?)",
    doctors
)

patients = [
    ('Jada Pinkett', 53, 'Female', 1, '9601234567'),
    ('Rita Ora', 33, 'Female', 2, '9612345678'),
    ('Ryan Gosling', 42, 'Male', 6, '9623456789'),
    ('Emma Stone', 36, 'Female', 7, '9634567890'),
    ('Rachel McAdams', 47, 'Female', 9, '9645678901'),
    ('Tom Holland', 29, 'Male', 6, '9656789012'),
    ('Zendaya Coleman', 27, 'Female', 7, '9667890123'),
    ('Chris Evans', 44, 'Male', 4, '9678901234'),
    ('Ana de Armas', 37, 'Female', 5, '9689012345'),
    ('John Cho', 51, 'Male', 9, '9690123456'),
    ('Priyanka Chopra', 42, 'Female', 2, '9701234567'),
    ('Dev Patel', 34, 'Male', 6, '9712345678'),
]
cur.executemany(
    "INSERT INTO Patients (patient_name, age, gender, doctor_id, contact_number) VALUES (?,?,?,?,?)",
    patients
)

appointments = [
    (1, 1, '2025-03-01 10:00', 'Completed'),
    (2, 2, '2025-03-02 11:30', 'Completed'),
    (3, 6, '2025-03-03 09:00', 'Pending'),
    (4, 7, '2025-03-04 15:00', 'Completed'),
    (5, 9, '2025-03-04 16:00', 'Cancelled'),
    (6, 6, '2025-03-05 10:30', 'Pending'),
    (7, 7, '2025-03-05 12:00', 'Completed'),
    (8, 4, '2025-03-06 09:30', 'Pending'),
    (9, 5, '2025-03-06 14:00', 'Completed'),
    (10, 9, '2025-03-07 11:00', 'Pending'),
    (3, 6, '2025-03-10 10:00', 'Completed'),
    (11, 2, '2025-03-11 13:00', 'Completed'),
    (12, 6, '2025-03-12 09:00', 'Pending'),
    (1, 1, '2025-03-15 10:00', 'Completed'),
    (7, 7, '2025-03-18 15:30', 'Cancelled'),
]
cur.executemany(
    "INSERT INTO Appointments (patient_id, doctor_id, appointment_date, status) VALUES (?,?,?,?)",
    appointments
)

conn.commit()
print(f"Inserted {len(doctors)} doctors, {len(patients)} patients, {len(appointments)} appointments")


In [ ]:
def q(sql, params=None):
    """Run a query and return a DataFrame."""
    return pd.read_sql_query(sql, conn, params=params)

print("Doctors:")
display(q("SELECT * FROM Doctors;"))
print("\nPatients:")
display(q("SELECT * FROM Patients;"))
print("\nAppointments:")
display(q("SELECT * FROM Appointments;"))


## 4. Graded Questions

### Q1 — Patients treated by doctors with more than 5 years of experience

In [ ]:
q1 = q('''
SELECT p.patient_id, p.patient_name, d.doctor_name, d.specialization, d.experience_years
FROM Patients p
JOIN Doctors d ON p.doctor_id = d.doctor_id
WHERE d.experience_years > 5
ORDER BY d.experience_years DESC;
''')
q1


### Q2 — Senior doctors (more than 10 years of experience)

In [ ]:
q2 = q('''
SELECT doctor_id, doctor_name, specialization, experience_years
FROM Doctors
WHERE experience_years > 10
ORDER BY experience_years DESC;
''')
q2


### Q3 — Doctors with no patients assigned (LEFT JOIN + NULL check)

In [ ]:
q3 = q('''
SELECT d.doctor_id, d.doctor_name, d.specialization
FROM Doctors d
LEFT JOIN Patients p ON d.doctor_id = p.doctor_id
WHERE p.patient_id IS NULL;
''')
q3


### Q4 — Number of appointments handled by each doctor

In [ ]:
q4 = q('''
SELECT d.doctor_id, d.doctor_name, COUNT(a.appointment_id) AS appointment_count
FROM Doctors d
LEFT JOIN Appointments a ON d.doctor_id = a.doctor_id
GROUP BY d.doctor_id, d.doctor_name
ORDER BY appointment_count DESC;
''')
q4


### Q5 — Aggregations, subqueries & window functions
Broken into five parts, matching the concepts called out in the README (`COUNT DISTINCT`,
`AVG` with `GROUP BY`, `RANK() OVER`, `COUNT() OVER (PARTITION BY ...)`, and a full outer-style
roster via `LEFT JOIN`).

In [ ]:
# 5a. Unique patients who have at least one appointment
q5a = q("SELECT COUNT(DISTINCT patient_id) AS unique_patients_with_appointments FROM Appointments;")
q5a


In [ ]:
# 5b. Average age of patients per doctor
q5b = q('''
SELECT d.doctor_id, d.doctor_name, ROUND(AVG(p.age), 1) AS avg_patient_age
FROM Doctors d
JOIN Patients p ON d.doctor_id = p.doctor_id
GROUP BY d.doctor_id, d.doctor_name
ORDER BY avg_patient_age DESC;
''')
q5b


In [ ]:
# 5c. Rank doctors by experience (window function)
q5c = q('''
SELECT doctor_id, doctor_name, experience_years,
       RANK() OVER (ORDER BY experience_years DESC) AS experience_rank
FROM Doctors;
''')
q5c


In [ ]:
# 5d. How many appointments fall on the same calendar day (window function)
q5d = q('''
SELECT appointment_id, appointment_date,
       COUNT(*) OVER (PARTITION BY DATE(appointment_date)) AS appointments_on_same_day
FROM Appointments
ORDER BY appointment_date;
''')
q5d


In [ ]:
# 5e. Full doctor -> patient roster (LEFT JOIN), including doctors with zero patients
q5e = q('''
SELECT d.doctor_id, d.doctor_name, d.specialization, d.experience_years,
       p.patient_id, p.patient_name, p.age, p.gender, p.contact_number
FROM Doctors d
LEFT JOIN Patients p ON d.doctor_id = p.doctor_id
ORDER BY d.doctor_id;
''')
q5e


### Bonus — Subquery example
Patients whose doctor has *above-average* experience across all doctors (correlated/subquery style,
often asked as a follow-up in interviews).

In [ ]:
bonus = q('''
SELECT p.patient_name, d.doctor_name, d.experience_years
FROM Patients p
JOIN Doctors d ON p.doctor_id = d.doctor_id
WHERE d.experience_years > (SELECT AVG(experience_years) FROM Doctors)
ORDER BY d.experience_years DESC;
''')
bonus


### Bonus — UPDATE / DELETE (DML)
The README also lists UPDATE/DELETE as a covered concept. Demonstrated safely on a copy-style
example: reschedule a pending appointment to completed, then clean up a cancelled one.

In [ ]:
# UPDATE example: mark an appointment as completed
cur.execute("UPDATE Appointments SET status = 'Completed' WHERE appointment_id = 3;")
conn.commit()
print("Updated appointment 3 ->", q("SELECT * FROM Appointments WHERE appointment_id = 3;").to_dict('records'))

# DELETE example: remove a cancelled appointment
cur.execute("DELETE FROM Appointments WHERE status = 'Cancelled' AND appointment_id = 5;")
conn.commit()
print("Remaining appointment count:", q("SELECT COUNT(*) AS n FROM Appointments;").iloc[0]['n'])


## 5. Visualizations

In [ ]:
fig, ax = plt.subplots()
exp = q("SELECT doctor_name, experience_years FROM Doctors ORDER BY experience_years DESC;")
ax.barh(exp['doctor_name'], exp['experience_years'], color='#2E86AB')
ax.set_xlabel("Years of Experience")
ax.set_title("Doctors Ranked by Experience")
ax.invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots()
appt_count = q('''
SELECT d.doctor_name, COUNT(a.appointment_id) AS appointment_count
FROM Doctors d LEFT JOIN Appointments a ON d.doctor_id = a.doctor_id
GROUP BY d.doctor_name
ORDER BY appointment_count DESC;
''')
ax.bar(appt_count['doctor_name'], appt_count['appointment_count'], color='#A23B72')
ax.set_ylabel("Appointments")
ax.set_title("Appointments Handled per Doctor")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots()
status = q("SELECT status, COUNT(*) AS n FROM Appointments GROUP BY status;")
ax.pie(status['n'], labels=status['status'], autopct='%1.0f%%',
       colors=['#2E86AB', '#F18F01', '#C73E1D'])
ax.set_title("Appointment Status Breakdown")
plt.tight_layout()
plt.show()


## 6. Export

The `hospital.db` SQLite file written in Section 1 is the artifact the deployment app (`app.py`)
connects to. We also export each result table to CSV for the report / README.

In [ ]:
import os
os.makedirs("exports", exist_ok=True)
q1.to_csv("exports/q1_patients_experienced_doctors.csv", index=False)
q2.to_csv("exports/q2_senior_doctors.csv", index=False)
q3.to_csv("exports/q3_doctors_no_patients.csv", index=False)
q4.to_csv("exports/q4_appointments_per_doctor.csv", index=False)
q5e.to_csv("exports/q5e_full_roster.csv", index=False)
print("CSV exports written to ./exports/")
print("Database file ready at:", os.path.abspath(DB_PATH))
conn.close()


## 7. Conclusion

- Built and populated a normalized 3-table hospital schema (`Doctors`, `Patients`, `Appointments`) with enforced foreign keys.
- Answered all graded questions using JOINs (INNER/LEFT), aggregates, `GROUP BY`, a correlated subquery, and window functions (`RANK()`, `COUNT() OVER PARTITION BY`).
- Fixed the orphaned foreign-key IDs present in the original `project_hospital.sql` draft so every query returns real, non-empty, sensible results.
- Exported a portable `hospital.db` file and CSV summaries, and shipped a Streamlit deployment app (`app.py`, in the same folder) so this project is a working, demoable artifact — not just a `.sql` file — for the GitHub repo and resume.

**Next step:** run `streamlit run app.py` to launch the interactive version.